# 1. 인물의 세계관 설정 분석

In [31]:
from pydantic import BaseModel
from typing import Literal, Optional


class LoreMappingResult(BaseModel):
    name: str
    gender: Literal["Male", "Female"]

    # 세계관 분석 결과
    historical_or_mythical: Literal["Historical", "Mythical", "Legendary", "Conceptual"]
    origin_country: Optional[str]
    era: str

    # Fate식 해석
    main_archetype: Literal[
        "Divine King",
        "Tyrant King",
        "Conqueror",
        "Saint",
        "Naval Commander",
        "Trickster",
        "Magus King",
        "Heroic Spirit"
    ]

    likely_class_candidates: list[str]

    # 전설성 / 신비도
    legend_rank: Literal["Low", "Medium", "High", "Extreme"]
    mystery_level: Literal["Modern", "Medieval", "Ancient", "Age of Gods"]

    # 신성 후보
    divinity_potential: Literal["None", "Low", "Medium", "High"]

    # 핵심 상징
    iconic_weapons_or_symbols: list[str]
    key_achievements: list[str]

    # strange Fake용 플래그
    suitable_for_pretender: bool
    suitable_for_foreigner: bool


In [32]:
from openai import OpenAI
from pydantic import ValidationError
from dotenv import load_dotenv

load_dotenv()

client = OpenAI()

def run_lore_mapping(name: str, gender: str) -> LoreMappingResult:
    system_prompt = """
You are a Fate/strange Fake and Fate/stay night lore analysis engine.
Analyze the given character according to Nasuverse rules.
Return ONLY valid JSON matching the provided schema.
Do NOT include extra commentary.
Only mark suitable_for_pretender as true if the character is known
for identity fraud, impersonation, false legends, or multiple historical identities.
"""

    user_prompt = f"""
Name: {name}
Gender: {gender}

Analyze this character for Fate/strange Fake universe adaptation.
"""

    # Responses API: 인자는 input, text_format (Chat Completions의 messages, response_format 아님)
    response = client.responses.parse(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        text_format=LoreMappingResult,
    )

    try:
        result: LoreMappingResult = response.output_parsed
        return result
    except ValidationError as e:
        print("Schema validation failed:", e)
        raise


# 2. 최적의 설정 탐색색

In [33]:
from pydantic import BaseModel
from typing import Literal


class OptimalServantSetup(BaseModel):
    # 핵심 Fate 스펙
    class_name: Literal[
        "Saber", "Archer", "Lancer", "Rider",
        "Caster", "Assassin", "Berserker",
        "Ruler", "Avenger", "Alter Ego",
        "Pretender", "Foreigner"
    ]
    spirit_origin_type: str
    attribute: Literal["Heaven", "Earth", "Human", "Star", "Beast"]
    alignment: str
    divinity_rank: str
    era: str

    # 전투/보구
    main_noble_phantasm_type: Literal[
        "Anti-Unit", "Anti-Army", "Anti-Fortress",
        "Anti-Country", "Reality Marble", "Support", "Conceptual"
    ]
    world_threat_level: Literal[
        "Local", "National", "Continental", "Mythic", "World-Class"
    ]

    # 설정 요약
    core_concept: str
    representative_noble_phantasm: str


In [34]:
from openai import OpenAI
from pydantic import ValidationError

client = OpenAI()

def run_optimal_servant_setup(lore: LoreMappingResult) -> OptimalServantSetup:
    system_prompt = """
You are a Fate/strange Fake character designer.
Using the provided lore mapping, generate ONE optimal Servant setup.

STRICT RULES:
- Do NOT generate any image prompts.
- Do NOT include character appearance or moe design.
- Focus ONLY on Fate/Nasuverse combat and lore configuration.
- Choose the most lore-accurate and optimal single Servant version.

Return ONLY valid JSON matching the schema.
"""

    user_prompt = f"""
Lore Mapping Result (JSON):

{lore.model_dump_json(indent=2)}

Generate the optimal single Servant configuration.
"""

    response = client.responses.parse(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        text_format=OptimalServantSetup,
    )

    try:
        result: OptimalServantSetup = response.output_parsed
        return result
    except ValidationError as e:
        print("Schema validation failed:", e)
        raise


# 3. 이미지 생성 프롬프트 출력력

In [35]:
from pydantic import BaseModel


class ImagePromptResult(BaseModel):
    # 최종 이미지 생성용 (영문)
    landscape_image_prompt_en: str

    # 네거티브/제거 옵션 (선택)
    negative_prompt_en: str


In [36]:
from openai import OpenAI
from pydantic import ValidationError

client = OpenAI()

def run_image_prompt_generator(
    lore: LoreMappingResult,
    servant: OptimalServantSetup,
) -> ImagePromptResult:
    system_prompt = """
You are an anime illustration prompt engineer for Fate-style characters.

TASK:
- Generate ONLY image generation prompts.
- Use moe-style character design.
- Respect the Servant class, era, and core concept.
- Apply natural Fate-style gender adaptation.
- Create a wide horizontal (16:9) cinematic composition.

STRICT RULES:
- Do NOT change lore or Servant settings.
- Do NOT add new Noble Phantasms.
- Do NOT include gameplay or combat stats.
- NO text, NO UI, NO watermark, NO logo, NO letters in image.

Return ONLY valid JSON matching the schema.
"""

    user_prompt = f"""
Lore Mapping (JSON):
{lore.model_dump_json(indent=2)}

Optimal Servant Setup (JSON):
{servant.model_dump_json(indent=2)}

Generate a Fate-style anime illustration prompt
for this Servant in landscape (16:9) format.
"""

    response = client.responses.parse(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        text_format=ImagePromptResult,
    )

    try:
        result: ImagePromptResult = response.output_parsed
        return result
    except ValidationError as e:
        print("Schema validation failed:", e)
        raise


In [37]:
def normalize_image_prompt(prompt: str) -> str:
    required_tokens = [
        "anime illustration",
        "Type-Moon official art style",
        "cinematic lighting",
        "dramatic background",
        "wide horizontal composition",
        "16:9 landscape",
        "no text",
        "no UI",
        "no watermark",
        "no logo",
        "no letters",
    ]

    normalized = prompt.strip()

    lower = normalized.lower()
    for token in required_tokens:
        if token.lower() not in lower:
            normalized += f", {token}"

    return normalized


In [38]:
def normalize_negative_prompt(negative: str) -> str:
    required_negative = [
        "text",
        "logo",
        "watermark",
        "UI",
        "signature",
        "subtitles",
        "low resolution",
        "blurry",
        "bad anatomy",
        "extra fingers",
        "extra arms",
        "deformed",
    ]

    normalized = negative.strip()
    lower = normalized.lower()

    for token in required_negative:
        if token.lower() not in lower:
            normalized += f", {token}"

    return normalized


In [39]:
def run_prompt_postprocess(step3: ImagePromptResult) -> ImagePromptResult:
    cleaned_prompt = normalize_image_prompt(step3.landscape_image_prompt_en)
    cleaned_negative = normalize_negative_prompt(step3.negative_prompt_en)

    return ImagePromptResult(
        landscape_image_prompt_en=cleaned_prompt,
        negative_prompt_en=cleaned_negative,
    )


# 4. 카드 데이터로 출력력

In [40]:
from pydantic import BaseModel
from typing import Literal


ElementTypeKR = Literal["불", "물", "땅", "바람", "빛", "어둠"]
ServantClassType = Literal[
    "Saber", "Archer", "Lancer", "Rider",
    "Caster", "Assassin", "Berserker",
    "Ruler", "Avenger", "Alter Ego",
    "Pretender", "Foreigner"
]


class ServantCardStats(BaseModel):
    # 카드 기본 정보 (한글)
    card_name: str                 # 카드명 (한글)
    card_type: ServantClassType   # 타입 = 클래스
    element: ElementTypeKR        # 속성 (불/물/땅/바람/빛/어둠)
    rarity: Literal[1, 2, 3, 4, 5] # 등급

    # 전투 스탯
    attack: int
    health: int

    # 보구(스킬) - 한글
    skill_name_1: str
    skill_desc_1: str   # 한줄

    skill_name_2: str
    skill_desc_2: str   # 한줄

    # 플레이버 텍스트 - 한줄
    flavor_text: str


In [ ]:
from openai import OpenAI
from pydantic import ValidationError

client = OpenAI()

def run_card_stat_generator(
    servant: OptimalServantSetup,
) -> ServantCardStats:
    system_prompt = """
당신은 Fate 스타일 서번트 카드 게임 디자이너입니다.

목표:
- 카드 결과물은 반드시 한글로 작성하십시오.
- 카드 타입은 서번트 클래스입니다.
- 속성은 반드시 다음 중 하나입니다: 불, 물, 땅, 바람, 빛, 어둠
- 스킬은 반드시 보구 또는 보구급 능력을 기반으로 합니다.
- 각 스킬 설명과 플레이버 텍스트는 반드시 한줄로 작성하십시오.
- 각 스킬 설명과 플레이버 텍스트(대사)는 각각 25자를 초과하지 않도록 작성하십시오.

규칙:
- card_type은 Servant 클래스와 반드시 일치해야 합니다.
- 카드명, 스킬명, 스킬설명, 플레이버 텍스트는 전부 한글이어야 합니다.
- 이미지 프롬프트나 추가 설정은 생성하지 마십시오.
- 게임 밸런스를 고려하여 등급에 맞는 수치를 설정하십시오.

밸런스 가이드:
- 1~2성: 낮음
- 3성: 중간
- 4성: 높음
- 5성: 최상급

반드시 스키마에 맞는 JSON만 출력하십시오.
"""

    user_prompt = f"""
최적 서번트 설정 (JSON):

{servant.model_dump_json(indent=2)}

위 설정을 기반으로 한글 서번트 카드 객체를 생성하십시오.
"""

    response = client.responses.parse(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        text_format=ServantCardStats,
    )

    try:
        result: ServantCardStats = response.output_parsed
        return result
    except ValidationError as e:
        print("Schema validation failed:", e)
        raise


In [42]:
# Step 1
step1_result = run_lore_mapping(
    name="길가메쉬",
    gender="여성"
)

print("=== Step 1: Lore Mapping ===")
print(step1_result.model_dump_json(indent=2))


# Step 2
step2_result = run_optimal_servant_setup(step1_result)

print("\n=== Step 2: Optimal Servant Setup ===")
print(step2_result.model_dump_json(indent=2))


# Step 3
step3_result = run_image_prompt_generator(step1_result, step2_result)

print("\n=== Step 3: Image Prompt ===")
print(step3_result.model_dump_json(indent=2))

# 3.5
step3_5 = run_prompt_postprocess(step3_result)

print('# Prompt')
print(' ' + step3_5.landscape_image_prompt_en)
print('# Negative Prompt')
print(' ' + step3_5.negative_prompt_en)

# Step 4
step4_result = run_card_stat_generator(step2_result)

print("\n=== Step 4: Servant Card Stats (KR) ===")
print(step4_result.model_dump_json(indent=2))


=== Step 1: Lore Mapping ===
{
  "name": "길가메쉬",
  "gender": "Female",
  "historical_or_mythical": "Mythical",
  "origin_country": "Mesopotamia",
  "era": "Ancient",
  "main_archetype": "Divine King",
  "likely_class_candidates": [
    "Saber",
    "Caster"
  ],
  "legend_rank": "High",
  "mystery_level": "Age of Gods",
  "divinity_potential": "High",
  "iconic_weapons_or_symbols": [
    "Epic of Gilgamesh",
    "Royal Scepter",
    "Bull of Heaven"
  ],
  "key_achievements": [
    "Slayed Humbaba",
    "Defeated the Bull of Heaven",
    "Sought the secret of immortality"
  ],
  "suitable_for_pretender": false,
  "suitable_for_foreigner": true
}

=== Step 2: Optimal Servant Setup ===
{
  "class_name": "Caster",
  "spirit_origin_type": "Mythical",
  "attribute": "Star",
  "alignment": "Lawful Good",
  "divinity_rank": "High",
  "era": "Ancient",
  "main_noble_phantasm_type": "Reality Marble",
  "world_threat_level": "Mythic",
  "core_concept": "Divine Kingship and Heroic Might",
  "repr